# MedCompress — Capacity Study
**Author: Abhishek Shekhar**

Tests 3 student architectures (Tiny/Small/Medium) at different capacities trained from scratch vs. with KD on ISIC 2020.

Disentangles capacity ceiling from genuine knowledge transfer.

In [ ]:
import os, sys, time, random, json, warnings
warnings.filterwarnings("ignore")
import numpy as np

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── GPU check ───────────────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → T4 GPU")
tf.config.set_visible_devices(gpus[0], 'GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)
print(f"TensorFlow {tf.__version__}  GPU: {gpus[0].name}")

# ── Directories (all outputs persist on Drive) ──────────────────────────────
EXP_NAME    = "capacity_study"   # ← overridden per notebook
DRIVE_BASE  = f"/content/drive/MyDrive/medcompress/capacity_study"
DATA_DIR    = f"/content/data/capacity_study"
RESULTS_DIR = f"{DRIVE_BASE}/results"
CKPT_DIR    = f"{DRIVE_BASE}/checkpoints"
MODELS_DIR  = f"{DRIVE_BASE}/models"
TFLITE_DIR  = f"{DRIVE_BASE}/tflite"
for d in [DRIVE_BASE, DATA_DIR, RESULTS_DIR, CKPT_DIR, MODELS_DIR, TFLITE_DIR]:
    os.makedirs(d, exist_ok=True)

SESSION_START = time.time()
print(f"Drive base : {DRIVE_BASE}")
print("Setup complete ✓")

In [ ]:
%%capture install_out
!pip install -q tensorflow-model-optimization tf2onnx onnx scikit-learn pillow tqdm
import tensorflow_model_optimization as tfmot
import tf2onnx
print("All packages installed ✓")

In [ ]:
# ── Kaggle credentials (run once per session) ──────────────────────────────
import os
KAGGLE_JSON = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(KAGGLE_JSON):
    from google.colab import files
    print("Upload your kaggle.json API token (from kaggle.com → Settings → API):")
    files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle credentials saved ✓")
else:
    print("Kaggle credentials already present ✓")

In [ ]:
# ── Data (ISIC 2020 — same as Notebook 1) ───────────────────────────────────
ISIC_DRIVE = "/content/drive/MyDrive/medcompress/isic_classification"
ISIC_DATA  = f"/content/data/capacity_study/isic2020"
os.makedirs(ISIC_DATA, exist_ok=True)

# Link from ISIC notebook data if already downloaded
if os.path.exists(f"{ISIC_DRIVE}/../../isic_classification") and    not os.path.exists(f"{ISIC_DATA}/train.csv"):
    # Try symlinking from existing ISIC download
    !ln -sf /content/data/isic_classification/isic2020 {ISIC_DATA} 2>/dev/null || true

if not os.path.exists(f"{ISIC_DATA}/train.csv"):
    print("Downloading ISIC 2020...")
    !kaggle competitions download -c siim-isic-melanoma-classification -p {ISIC_DATA}
    !cd {ISIC_DATA} && unzip -q "*.zip" && rm -f *.zip

import pandas as pd
df=pd.read_csv(f"{ISIC_DATA}/train.csv")
jpeg_dir=f"{ISIC_DATA}/jpeg/train"
if not os.path.exists(jpeg_dir): jpeg_dir=f"{ISIC_DATA}/train"
df["image_path"]=df["image_name"].apply(lambda x: f"{jpeg_dir}/{x}.jpg")
df=df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
counts=df["target"].value_counts()
CLASS_WEIGHTS={0:len(df)/(2*counts[0]),1:len(df)/(2*counts[1])}
print(f"Dataset: {len(df)} samples")

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32
SEEDS      = [42, 123, 456]  # 3 seeds for capacity study
from tensorflow import keras; from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

train_df,test_df=train_test_split(df,test_size=0.15,stratify=df["target"],random_state=42)
train_df,val_df =train_test_split(train_df,test_size=0.15/0.85,stratify=train_df["target"],random_state=42)

def make_ds(dataframe,augment=False,shuffle=False):
    paths=dataframe["image_path"].values; labels=dataframe["target"].values.astype(np.float32)
    def load(path,label):
        img=tf.image.decode_jpeg(tf.io.read_file(path),channels=3)
        img=tf.image.resize(img,[IMG_SIZE,IMG_SIZE])
        return tf.cast(img,tf.float32)/127.5-1.0, label
    def aug(img,label):
        img=tf.image.random_flip_left_right(img); img=tf.image.random_brightness(img,0.2)
        return img,label
    ds=tf.data.Dataset.from_tensor_slices((paths,labels))
    if shuffle: ds=ds.shuffle(len(paths),reshuffle_each_iteration=True)
    ds=ds.map(load,num_parallel_calls=tf.data.AUTOTUNE)
    if augment: ds=ds.map(aug,num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_ds(train_df,True,True); val_ds=make_ds(val_df); test_ds=make_ds(test_df)

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def compute_cls_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob.ravel() >= thr).astype(int)
    y_true = y_true.astype(int).ravel()
    auc  = roc_auc_score(y_true, y_prob.ravel())
    f1   = f1_score(y_true, y_pred, zero_division=0)
    sens = recall_score(y_true, y_pred, zero_division=0)
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return {"auc":float(auc),"f1":float(f1),"sensitivity":float(sens),"specificity":float(spec)}

def export_tflite(model, path, quant="fp32", calib_ds=None, n_calib=200):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quant == "int8":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if calib_ds:
            samples = []
            for imgs, _ in calib_ds:
                for i in range(len(imgs)):
                    samples.append(imgs[i].numpy())
                    if len(samples) >= n_calib: break
                if len(samples) >= n_calib: break
            def rep():
                for s in samples: yield [np.expand_dims(s,0)]
            conv.representative_dataset = rep
            conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            conv.inference_input_type  = tf.uint8
            conv.inference_output_type = tf.uint8
    elif quant == "fp16":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]
    data = conv.convert()
    with open(path,"wb") as f: f.write(data)
    mb = os.path.getsize(path)/1e6
    print(f"  TFLite {quant}: {os.path.basename(path)} ({mb:.1f} MB)")
    return mb

def eval_tflite(path, dataset, warmup=5):
    interp = tf.lite.Interpreter(model_path=path, num_threads=4)
    interp.allocate_tensors()
    inp_d = interp.get_input_details()[0]
    out_d = interp.get_output_details()[0]
    preds, labels, lats = [], [], []
    for imgs, lbls in dataset:
        for i in range(len(imgs)):
            x = np.expand_dims(imgs[i].numpy(), 0)
            if inp_d["dtype"] == np.uint8:
                sc,zp = inp_d["quantization"]; x=(x/sc+zp).astype(np.uint8)
            interp.set_tensor(inp_d["index"], x)
            t0=time.perf_counter(); interp.invoke(); t1=time.perf_counter()
            lats.append((t1-t0)*1000)
            r=interp.get_tensor(out_d["index"])
            if out_d["dtype"]==np.uint8:
                sc,zp=out_d["quantization"]; r=(r.astype(np.float32)-zp)*sc
            preds.append(r.squeeze()); labels.append(lbls[i].numpy())
    lats = np.array(lats[warmup:])
    m = compute_cls_metrics(np.array(labels), np.array(preds))
    m["latency_ms"] = float(np.median(lats))
    m["latency_p95_ms"] = float(np.percentile(lats,95))
    m["size_mb"] = os.path.getsize(path)/1e6
    print(f"  TFLite AUC={m['auc']:.4f}  lat={m['latency_ms']:.1f}ms  sz={m['size_mb']:.1f}MB")
    return m

def done_flag(name): return f"{RESULTS_DIR}/{name}.done"
def is_done(name): return os.path.exists(done_flag(name))
def mark_done(name, result):
    with open(f"{RESULTS_DIR}/{name}.json","w") as f: json.dump(result, f)
    open(done_flag(name),"w").close()
def load_result(name):
    with open(f"{RESULTS_DIR}/{name}.json") as f: return json.load(f)

def train_model(model, train_ds, val_ds, cfg, name, monitor="val_auc", mode="max"):
    """Train with Drive checkpoint + resume. Returns history."""
    ckpt = f"{CKPT_DIR}/{name}_best.keras"
    epoch_f = f"{CKPT_DIR}/{name}_epoch.json"
    initial_epoch = 0
    if os.path.exists(ckpt):
        try:
            model.load_weights(ckpt)
            if os.path.exists(epoch_f):
                initial_epoch = json.load(open(epoch_f)).get("epoch",0)
            print(f"  Resumed from epoch {initial_epoch} ✓")
        except Exception as e:
            print(f"  Checkpoint load failed ({e}), starting fresh")
    class _EpochLog(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            json.dump({"epoch":epoch+1}, open(epoch_f,"w"))
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt, save_best_only=True, monitor=monitor, mode=mode, verbose=0),
        tf.keras.callbacks.EarlyStopping(
            patience=cfg.get("patience",7), monitor=monitor, mode=mode,
            restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5, patience=3, mode=mode, verbose=0),
        _EpochLog()
    ]
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=cfg.get("epochs",20), initial_epoch=initial_epoch,
        class_weight=cfg.get("class_weight"), callbacks=cbs, verbose=1)
    return model, history

print("Utilities loaded ✓")

In [ ]:
# ── Student architectures at 3 capacities ───────────────────────────────────
STUDENT_CONFIGS = {
    "tiny":   {"filters":8,  "dense":32,  "params_approx":"~0.1M"},
    "small":  {"filters":16, "dense":64,  "params_approx":"~0.4M"},
    "medium": {"filters":32, "dense":128, "params_approx":"~1.5M"},
}

def build_cnn_student(cfg, name):
    """Custom CNN student at specified capacity."""
    f,d=cfg["filters"],cfg["dense"]
    inp=keras.Input((IMG_SIZE,IMG_SIZE,3))
    x=layers.Conv2D(f,3,strides=2,padding="same",use_bias=False)(inp)
    x=layers.BatchNormalization()(x); x=layers.ReLU()(x)
    for mult in [1,2,4]:
        x=layers.DepthwiseConv2D(3,strides=2,padding="same",use_bias=False)(x)
        x=layers.BatchNormalization()(x); x=layers.ReLU()(x)
        x=layers.Conv2D(f*mult,1,use_bias=False)(x)
        x=layers.BatchNormalization()(x); x=layers.ReLU()(x)
    x=layers.GlobalAveragePooling2D()(x); x=layers.Dropout(0.3)(x)
    x=layers.Dense(d,activation="relu")(x); x=layers.Dropout(0.2)(x)
    m=keras.Model(inp,layers.Dense(1,activation="sigmoid")(x),name=name)
    print(f"  {name}: {m.count_params():,} params")
    return m

# Teacher: load from ISIC experiment (or train if not available)
teacher_path=f"/content/drive/MyDrive/medcompress/isic_classification/models/efficientnetb0_s42.keras"
if os.path.exists(teacher_path):
    teacher=keras.models.load_model(teacher_path); print("Teacher loaded from ISIC NB ✓")
else:
    print("Training teacher from scratch for capacity study...")
    bb=keras.applications.EfficientNetB0(include_top=False,weights="imagenet",input_shape=(224,224,3))
    for l in bb.layers[:-20]: l.trainable=False
    inp=keras.Input((224,224,3)); x=bb(inp,training=False)
    x=layers.GlobalAveragePooling2D()(x); x=layers.BatchNormalization()(x)
    x=layers.Dropout(0.3)(x); x=layers.Dense(256,activation="relu")(x)
    teacher=keras.Model(inp,layers.Dense(1,activation="sigmoid")(x))
    teacher.compile(keras.optimizers.Adam(1e-4),"binary_crossentropy",[keras.metrics.AUC(name="auc")])
    teacher.fit(train_ds,validation_data=val_ds,epochs=15,class_weight=CLASS_WEIGHTS,verbose=1)
    teacher.save(f"{MODELS_DIR}/teacher_cap_study.keras")

print("Ready ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CAPACITY STUDY — Scratch vs KD for each architecture × seed
# ══════════════════════════════════════════════════════════════════════════════
T=4.0; ALPHA=0.7
all_results=[]

for cap_name, cap_cfg in STUDENT_CONFIGS.items():
    for seed in SEEDS:
        set_seed(seed)
        # ─ Scratch ─
        key=f"{cap_name}_scratch_s{seed}"
        if is_done(key):
            r=load_result(key); all_results.append(r); print(f"⏩ {key} AUC={r['auc']:.4f}")
        else:
            m=build_cnn_student(cap_cfg, f"{cap_name}_s{seed}")
            m.compile(keras.optimizers.Adam(1e-4),"binary_crossentropy",[keras.metrics.AUC(name="auc")])
            m,_=train_model(m,train_ds,val_ds,{"epochs":25,"patience":8,"class_weight":CLASS_WEIGHTS},
                            key,"val_auc")
            r=eval_model(m,test_ds,key); r["capacity"]=cap_name; r["condition"]="scratch"; r["seed"]=seed
            r["params"]=m.count_params(); mark_done(key,r); all_results.append(r)
        # ─ KD ─
        key_kd=f"{cap_name}_kd_s{seed}"
        if is_done(key_kd):
            r=load_result(key_kd); all_results.append(r); print(f"⏩ {key_kd} AUC={r['auc']:.4f}")
        else:
            student=build_cnn_student(cap_cfg, f"{cap_name}_kd_s{seed}")
            opt=keras.optimizers.Adam(1e-4); best_auc,patience_c=0,0
            ckpt=f"{CKPT_DIR}/{key_kd}.keras"
            if os.path.exists(ckpt): student.load_weights(ckpt)
            for epoch in range(25):
                for imgs,lbls in train_ds:
                    with tf.GradientTape() as tape:
                        t_out=teacher(imgs,training=False); s_out=student(imgs,training=True)
                        def soft(x): return tf.nn.sigmoid(tf.math.log(x/(1-x+1e-7)+1e-7)/T)
                        ts,ss=soft(t_out),soft(s_out); eps=1e-7
                        kl=ts*tf.math.log((ts+eps)/(ss+eps))+(1-ts)*tf.math.log((1-ts+eps)/(1-ss+eps))
                        loss=ALPHA*tf.reduce_mean(kl)*(T**2)+(1-ALPHA)*tf.reduce_mean(
                            keras.losses.binary_crossentropy(tf.expand_dims(lbls,-1),s_out))
                    opt.apply_gradients(zip(tape.gradient(loss,student.trainable_variables),student.trainable_variables))
                val_auc=eval_model(student,val_ds,f"ep{epoch+1}")["auc"]
                if val_auc>best_auc: best_auc=val_auc; patience_c=0; student.save_weights(ckpt)
                else:
                    patience_c+=1
                    if patience_c>=8: break
            student.load_weights(ckpt)
            r=eval_model(student,test_ds,key_kd)
            r["capacity"]=cap_name; r["condition"]="kd"; r["seed"]=seed; r["params"]=student.count_params()
            mark_done(key_kd,r); all_results.append(r)

table=pd.DataFrame(all_results)
table.to_csv(f"{RESULTS_DIR}/capacity_study.csv",index=False)
pivot=table.groupby(["capacity","condition"])["auc"].agg(["mean","std"]).round(4)
print("\nCapacity Study Results:")
print(pivot)
print(f"\nSaved: {RESULTS_DIR}")